# Persona Fairness — Token Usage & Cost Summary

Summarises token usage and estimated cost for the **four reported clinician-model runs** of the persona-based fairness experiment (Gemini 3.5 Flash, GPT-5.5, GPT-5.4-mini, Claude Opus 4.6).

Each run directory under `../../fairness/results/runs` contains a `cost_summary.json` recording per-node token usage and USD cost. Each figure aggregates the **clinician model under test** with the **GPT-5.5 patient simulator** that generates the patient turns (the `stop_check_node` is reported as `unknown` with zero cost).

This notebook is the analysis companion to the **Computational Cost** section of Supplementary Material 7. `supplements/7_fairness/build_report.py` (`render_cost_table`) is the source of truth for the tables in the PDF; the run IDs below match `MODEL_RUNS` there and the `results_*.ipynb` notebooks. Unit pricing is read from `eval/_utils.py` (`_PRICING`), the same table used to compute `cost_summary.json`.

In [1]:
import json
import re
from pathlib import Path

import pandas as pd

# --- Configuration ---
# Display name -> run id. Order matches the canonical model order in
# Supplementary Material 7 (Claude last; it lacks probability metrics).
RUNS = {
    "Gemini 3.5 Flash": "persona_20260601_183211_d415d86a",
    "GPT-5.5": "persona_20260604_135655_333650b8",
    "GPT-5.4-mini": "persona_20260521_172943_d0326dbc",
    "Claude Opus 4.6": "persona_20260601_142913_bf3804f5",
}
RUNS_DIR = Path("../../fairness/results/runs")
PRICING_PY = Path("../../eval/_utils.py")
SIM_MODEL_ID = "gpt-5.5-2026-04-23"  # patient simulator model

TOKEN_KEYS = ("input_tokens", "cached_tokens", "output_tokens", "reasoning_tokens", "calls", "cost_usd")


def load_cost(run_id):
    with open(RUNS_DIR / run_id / "cost_summary.json") as f:
        return json.load(f)


def split_doctor_simulator(cost):
    """Aggregate per-node costs into doctor vs patient-simulator buckets."""
    buckets = {"doctor": {k: 0 for k in TOKEN_KEYS}, "simulator": {k: 0 for k in TOKEN_KEYS}}
    for node in cost.get("per_node", []):
        b = buckets["doctor"] if node["node"] == "doctor" else buckets["simulator"]
        for k in TOKEN_KEYS:
            b[k] += node.get(k, 0)
    return buckets


def doctor_model_id(cost):
    """model_id of the doctor node in a run's cost summary."""
    for node in cost.get("per_node", []):
        if node["node"] == "doctor":
            return node.get("model_id", "unknown")
    return "unknown"


def load_pricing():
    """Parse _PRICING (USD per 1M tokens) from eval/_utils.py."""
    src = PRICING_PY.read_text()
    m = re.search(r"_PRICING[^=]*=\s*\{(.*?)\n\}", src, re.DOTALL)
    pricing = {}
    if m:
        for model_id, body in re.findall(r'"([^"]+)":\s*\{([^}]*)\}', m.group(1)):
            pricing[model_id] = {k: float(v) for k, v in re.findall(r'"(\w+)":\s*([\d.]+)', body)}
    return pricing


costs = {model: load_cost(run_id) for model, run_id in RUNS.items()}
splits = {model: split_doctor_simulator(c) for model, c in costs.items()}
pricing = load_pricing()
for model, run_id in RUNS.items():
    print(f"{model}: loaded {run_id}")

Gemini 3.5 Flash: loaded persona_20260601_183211_d415d86a
GPT-5.5: loaded persona_20260604_135655_333650b8
GPT-5.4-mini: loaded persona_20260521_172943_d0326dbc
Claude Opus 4.6: loaded persona_20260601_142913_bf3804f5


## Unit pricing (USD per 1M tokens)

Provider list prices used to compute the costs, for the clinician models under test and the GPT-5.5 patient simulator. Uncached input, cached input, and output tokens are billed separately.

In [2]:
price_index = {model: doctor_model_id(costs[model]) for model in RUNS}
price_index["Patient simulator"] = SIM_MODEL_ID

price_tbl = pd.DataFrame(
    {
        label: {
            "Model ID": model_id,
            "Input ($/M)": pricing.get(model_id, {}).get("input", float("nan")),
            "Cached ($/M)": pricing.get(model_id, {}).get("cached", float("nan")),
            "Output ($/M)": pricing.get(model_id, {}).get("output", float("nan")),
        }
        for label, model_id in price_index.items()
    }
).T
display(price_tbl)

,Model ID,Input ($/M),Cached ($/M),Output ($/M)
Gemini 3.5 Flash,gemini-3.5-flash,1.5,0.0,9.0
GPT-5.5,gpt-5.5-2026-04-23,5.0,0.5,30.0
GPT-5.4-mini,gpt-5.4-mini-2026-03-17,0.75,0.075,4.5
Claude Opus 4.6,claude-opus-4-6,5.0,0.5,25.0
Patient simulator,gpt-5.5-2026-04-23,5.0,0.5,30.0


## Token usage (millions)

Totals across the clinician model and patient simulator. Cached tokens are a subset of input tokens served from the provider prompt cache; reasoning tokens are billed output tokens spent on internal reasoning.

In [3]:
usage = pd.DataFrame(
    {
        model: {
            "Input (M)": costs[model]["total"]["input_tokens"] / 1e6,
            "Cached (M)": costs[model]["total"]["cached_tokens"] / 1e6,
            "Cache hit (%)": costs[model].get("cache_hit_rate_pct", 0.0),
            "Output (M)": costs[model]["total"]["output_tokens"] / 1e6,
            "Reasoning (M)": costs[model]["total"]["reasoning_tokens"] / 1e6,
            "API calls": costs[model]["total"]["calls"],
        }
        for model in RUNS
    }
).T
usage["API calls"] = usage["API calls"].astype(int)
display(usage.round(1))

,Input (M),Cached (M),Cache hit (%),Output (M),Reasoning (M),API calls
Gemini 3.5 Flash,382.2,186.8,48.9,25.4,6.6,221160
GPT-5.5,377.6,169.5,44.9,39.2,19.2,221160
GPT-5.4-mini,310.0,146.6,47.3,33.9,18.6,179256
Claude Opus 4.6,305.6,141.9,46.4,18.9,6.1,137352


## Cost (USD)

Split between the clinician model under evaluation and the GPT-5.5 patient-simulator overhead. Costs use the unit pricing above. Totals are not directly comparable across models: Claude Opus 4.6 uses a single assessment per conversation (no repeats), so its counts are lower.

In [4]:
cost = pd.DataFrame(
    {
        model: {
            "Clinician ($)": splits[model]["doctor"]["cost_usd"],
            "Patient simulator ($)": splits[model]["simulator"]["cost_usd"],
            "Total ($)": costs[model]["total"]["cost_usd"],
        }
        for model in RUNS
    }
).T
display(cost.round(2))

,Clinician ($),Patient simulator ($),Total ($)
Gemini 3.5 Flash,123.00,1422.85,1545.85
GPT-5.5,862.23,1440.58,2302.82
GPT-5.4-mini,106.38,1198.64,1305.03
Claude Opus 4.6,147.47,1295.50,1442.97


## Per-node breakdown

Full per-node detail for each run, for reference.

In [5]:
for model in RUNS:
    print(f"=== {model} ===")
    nodes = pd.DataFrame(costs[model]["per_node"]).set_index("node")
    display(nodes)

=== Gemini 3.5 Flash ===


,model_id,input_tokens,cached_tokens,output_tokens,reasoning_tokens,calls,cost_usd
node,,,,,,,
communication_interface_node,gpt-5.5-2026-04-23,97713635,56942592,1905696,317611,29100,289.4974
content_node,gpt-5.5-2026-04-23,117302935,84370944,7633595,5133977,29100,435.8533
doctor,gemini-3.5-flash,38495412,0,7250833,0,87300,123.0006
feelings_node,gpt-5.5-2026-04-23,30018462,10913536,993057,72381,11640,130.7731
ingest,gpt-5.5-2026-04-23,18536910,5232128,3613137,809668,5820,177.5341
stop_check_node,unknown,0,0,0,0,29100,0.0000
strategy_node,gpt-5.5-2026-04-23,80101495,29328384,4022045,310697,29100,389.1911


=== GPT-5.5 ===


,model_id,input_tokens,cached_tokens,output_tokens,reasoning_tokens,calls,cost_usd
node,,,,,,,
communication_interface_node,gpt-5.5-2026-04-23,96305277,55651328,1763940,259268,29100,284.0136
content_node,gpt-5.5-2026-04-23,116171918,83484160,7005957,4571493,29100,415.3596
doctor,gpt-5.5-2026-04-23,38141117,0,22384312,13611271,87300,862.2349
feelings_node,gpt-5.5-2026-04-23,29495237,7077888,973341,67650,11640,144.8259
ingest,gpt-5.5-2026-04-23,18536910,6418688,3106641,353160,5820,156.9997
stop_check_node,unknown,0,0,0,0,29100,0.0000
strategy_node,gpt-5.5-2026-04-23,78970478,16839424,4010323,384582,29100,439.3847


=== GPT-5.4-mini ===


,model_id,input_tokens,cached_tokens,output_tokens,reasoning_tokens,calls,cost_usd
node,,,,,,,
communication_interface_node,gpt-5.5-2026-04-23,77602969,44744960,1744503,437867,23280,238.9976
content_node,gpt-5.5-2026-04-23,93135684,67307520,6259753,4198836,23280,350.5872
doctor,gpt-5.4-mini-2026-03-17,31132054,0,18452364,12621374,69840,106.3847
feelings_node,gpt-5.5-2026-04-23,29529148,9375488,985654,72348,11640,135.0257
ingest,gpt-5.5-2026-04-23,14829528,3288320,3095940,859916,4656,152.2284
stop_check_node,unknown,0,0,0,0,23280,0.0000
strategy_node,gpt-5.5-2026-04-23,63721404,21932288,3396454,368094,23280,321.8053


=== Claude Opus 4.6 ===


,model_id,input_tokens,cached_tokens,output_tokens,reasoning_tokens,calls,cost_usd
node,,,,,,,
communication_interface_node,gpt-5.5-2026-04-23,81177603,44928000,1903681,435919,23280,260.8224
content_node,gpt-5.5-2026-04-23,95974872,67172608,7191198,4876161,23280,393.3336
doctor,claude-opus-4-6,16154926,0,2667740,0,27936,147.4681
feelings_node,gpt-5.5-2026-04-23,30911951,9197824,992315,48783,11640,142.9390
ingest,gpt-5.5-2026-04-23,14829528,3404032,2809057,578699,4656,143.1012
stop_check_node,unknown,0,0,0,0,23280,0.0000
strategy_node,gpt-5.5-2026-04-23,66560592,17165056,3324820,153565,23280,355.3048


## LaTeX for Supplementary Material 7

Mirrors `render_cost_table()` in `supplements/7_fairness/build_report.py` (the source of truth for the PDF). Use this to cross-check the rendered tables.

In [6]:
def m(x):
    return f"{x / 1e6:.1f}"


def price(x):
    s = f"{x:.3f}".rstrip("0")
    intp, _, frac = s.partition(".")
    return f"{intp}.{(frac + '00')[:2]}" if len(frac) < 2 else f"{intp}.{frac}"


price_rows = "\n".join(
    f"{label} & \\texttt{{{model_id}}} & "
    f"{price(pricing.get(model_id, {}).get('input', 0))} & "
    f"{price(pricing.get(model_id, {}).get('cached', 0))} & "
    f"{price(pricing.get(model_id, {}).get('output', 0))} \\\\"
    for label, model_id in price_index.items()
)

usage_rows = "\n".join(
    f"{model} & {m(costs[model]['total']['input_tokens'])} & "
    f"{m(costs[model]['total']['cached_tokens'])} & "
    f"{costs[model].get('cache_hit_rate_pct', 0.0):.1f} & "
    f"{m(costs[model]['total']['output_tokens'])} & "
    f"{m(costs[model]['total']['reasoning_tokens'])} & "
    f"{costs[model]['total']['calls']:,} \\\\"
    for model in RUNS
)

cost_rows = "\n".join(
    f"{model} & {splits[model]['doctor']['cost_usd']:,.0f} & "
    f"{splits[model]['simulator']['cost_usd']:,.0f} & "
    f"{costs[model]['total']['cost_usd']:,.0f} \\\\"
    for model in RUNS
)

print("% --- Table A: unit pricing (USD per 1M tokens) ---")
print(price_rows)
print("\n% --- Table B: token usage ---")
print(usage_rows)
print("\n% --- Table C: cost (USD) ---")
print(cost_rows)

% --- Table A: unit pricing (USD per 1M tokens) ---
Gemini 3.5 Flash & \texttt{gemini-3.5-flash} & 1.50 & 0.00 & 9.00 \\
GPT-5.5 & \texttt{gpt-5.5-2026-04-23} & 5.00 & 0.50 & 30.00 \\
GPT-5.4-mini & \texttt{gpt-5.4-mini-2026-03-17} & 0.75 & 0.075 & 4.50 \\
Claude Opus 4.6 & \texttt{claude-opus-4-6} & 5.00 & 0.50 & 25.00 \\
Patient simulator & \texttt{gpt-5.5-2026-04-23} & 5.00 & 0.50 & 30.00 \\

% --- Table B: token usage ---
Gemini 3.5 Flash & 382.2 & 186.8 & 48.9 & 25.4 & 6.6 & 221,160 \\
GPT-5.5 & 377.6 & 169.5 & 44.9 & 39.2 & 19.2 & 221,160 \\
GPT-5.4-mini & 310.0 & 146.6 & 47.3 & 33.9 & 18.6 & 179,256 \\
Claude Opus 4.6 & 305.6 & 141.9 & 46.4 & 18.9 & 6.1 & 137,352 \\

% --- Table C: cost (USD) ---
Gemini 3.5 Flash & 123 & 1,423 & 1,546 \\
GPT-5.5 & 862 & 1,441 & 2,303 \\
GPT-5.4-mini & 106 & 1,199 & 1,305 \\
Claude Opus 4.6 & 147 & 1,296 & 1,443 \\
